# QuantBench Model Gate Scan v1

This notebook screens existing SOXL model replay CSVs against a QuantBench-style portfolio gate: each required regime window must have strategy max drawdown below 20% and strategy return above Buy&Hold.

The scan intentionally avoids judging only from the earliest/full replay window because SOXL volatility regimes differ sharply across 2020, 2022, and 2023-2024.

## Governance

- Direction: `model_selection`
- Role: `diagnose`
- Changed layer: `selection` / benchmark evaluation, not a new actor training run
- External reference: QuantBench evaluation framing: portfolio return, max drawdown, robustness across market conditions
- Gate: required windows `test_2022` and `test_2023_2024`; each must satisfy `max_drawdown_pct < 20` and `strategy_return_pct > buyhold_return_pct`
- Decision from this card cannot promote a model by itself; it only identifies whether the hard objective is already met by existing replay artifacts.

In [1]:
from pathlib import Path
import json
import sys
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "src" / "soxl_quant").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

CARD = ROOT / "runtime/model_selection/card/quantbench_model_gate_scan_v1"
from src.soxl_quant.app.quantbench_gates import scan_model_replays, write_scan_outputs

CARD.mkdir(parents=True, exist_ok=True)

In [2]:
result = scan_model_replays(
    root=ROOT,
    required_windows=("test_2022", "test_2023_2024"),
    max_drawdown_pct=20.0,
)
write_scan_outputs(result, output_dir=CARD)
result["summary"]

{'profiles_scanned': 39,
 'rows': 200,
 'required_windows': ['test_2022', 'test_2023_2024'],
 'max_drawdown_pct': 20.0,
 'passing_profiles': 0}

In [3]:
rows = pd.DataFrame(result["rows"])
# Rank by hard-gate relevance: fewer failures, lower drawdown, higher relative return.
view = rows.loc[rows["window"].isin(result["summary"]["required_windows"])].copy()
view["failure_count"] = view["failures"].map(len)
ranked = view.sort_values(["failure_count", "max_drawdown_pct", "relative_return_pct"], ascending=[True, True, False])
ranked.head(20)

,profile,path,window,rows,start,end,strategy_return_pct,buyhold_return_pct,relative_return_pct,max_drawdown_pct,gate_pass,failures,failure_count
112,drawdown_rebound_reward_train_v1_wf_2022,runtime/model/card/drawdown_rebound_reward_tra...,test_2022,250,2022-01-03,2022-12-29,-7.000281,-86.588072,79.587791,9.875679,True,[],0
132,mode_conditioned_rebound_reward_train_v2_wf_2022,runtime/model/card/mode_conditioned_rebound_re...,test_2022,250,2022-01-03,2022-12-29,-12.705282,-86.588072,73.882789,15.900697,True,[],0
22,balanced_rule_residual_imitation_v4_wf_2020,runtime/model/card/balanced_rule_residual_imit...,test_2022,0,None,None,0.000000,0.000000,0.000000,0.000000,False,[empty_window],1
23,balanced_rule_residual_imitation_v4_wf_2020,runtime/model/card/balanced_rule_residual_imit...,test_2023_2024,0,None,None,0.000000,0.000000,0.000000,0.000000,False,[empty_window],1
28,balanced_rule_residual_imitation_v4_wf_2022,runtime/model/card/balanced_rule_residual_imit...,test_2023_2024,0,None,None,0.000000,0.000000,0.000000,0.000000,False,[empty_window],1
32,balanced_rule_residual_imitation_v4_wf_2023_2024,runtime/model/card/balanced_rule_residual_imit...,test_2022,0,None,None,0.000000,0.000000,0.000000,0.000000,False,[empty_window],1
42,balanced_rule_residual_mode_v1_wf_2020,runtime/model/card/balanced_rule_residual_mode...,test_2022,0,None,None,0.000000,0.000000,0.000000,0.000000,False,[empty_window],1
43,balanced_rule_residual_mode_v1_wf_2020,runtime/model/card/balanced_rule_residual_mode...,test_2023_2024,0,None,None,0.000000,0.000000,0.000000,0.000000,False,[empty_window],1
48,balanced_rule_residual_mode_v1_wf_2022,runtime/model/card/balanced_rule_residual_mode...,test_2023_2024,0,None,None,0.000000,0.000000,0.000000,0.000000,False,[empty_window],1
52,balanced_rule_residual_mode_v1_wf_2023_2024,runtime/model/card/balanced_rule_residual_mode...,test_2022,0,None,None,0.000000,0.000000,0.000000,0.000000,False,[empty_window],1


In [4]:
passing = result["passing_profiles"]
decision = "hard_gate_pass_found" if passing else "hard_gate_no_existing_model_passes"
summary = {
    **result,
    "experiment": "quantbench_model_gate_scan_v1",
    "decision": decision,
    "promotion_allowed": bool(passing),
    "training_allowed_from_this_card": False,
    "next_action": "If no pass, train or compose candidates explicitly against 2022 drawdown control and 2023-2024 participation, then rerun this scan.",
    "sidecars": ["summary.json", "model_gate_scan.csv"],
}
(CARD / "summary.json").write_text(json.dumps(summary, indent=2, sort_keys=True), encoding="utf-8")
summary["decision"], summary["passing_profiles"][:10]

('hard_gate_no_existing_model_passes', [])

## Conclusion

The current evidence should be read from the executed `summary.json` and `model_gate_scan.csv` sidecars. If no existing model passes both required windows, the next productive step is not another broad dashboard inspection; it is a targeted candidate loop that explicitly trades off 2022 drawdown control against 2023-2024 Buy&Hold participation, then reruns this exact gate.